In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import datetime, pandas

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Union
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import warnings
from abc import ABC, abstractmethod

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [2]:
import h2o
print(h2o.__version__)
from h2o.automl import H2OAutoML

h2o.init(max_mem_size='8G')

3.46.0.7
Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.27" 2025-04-15; OpenJDK Runtime Environment (build 11.0.27+6-post-Ubuntu-0ubuntu122.04); OpenJDK 64-Bit Server VM (build 11.0.27+6-post-Ubuntu-0ubuntu122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.11/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmp35vo7s3f
  JVM stdout: /tmp/tmp35vo7s3f/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmp35vo7s3f/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,5 months and 26 days
H2O_cluster_name:,H2O_from_python_unknownUser_xx35nf
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,8 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


In [3]:
train = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv').dropna()
test = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/test.csv').dropna()

In [4]:
train.columns

Index(['date_id', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1',
       'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19',
       'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'I1', 'I2', 'I3',
       'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M1', 'M10', 'M11', 'M12', 'M13',
       'M14', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7',
       'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5',
       'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4',
       'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V10', 'V11', 'V12', 'V13', 'V2',
       'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'forward_returns',
       'risk_free_rate', 'market_forward_excess_returns'],
      dtype='object')

In [5]:
train

,date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,V3,V4,V5,V6,V7,V8,V9,forward_returns,risk_free_rate,market_forward_excess_returns
6969,6969,0,0,0,0,0,-1,0,0,0,...,0.976852,0.996693,0.534651,0.884921,-0.911387,0.979167,-0.847754,0.001145,0.000040,0.000797
6970,6970,0,0,0,0,0,0,0,1,0,...,0.986772,0.997354,0.283256,0.769180,-0.842354,0.970238,-0.821913,0.004738,0.000040,0.004390
6971,6971,0,0,0,0,1,0,0,1,0,...,0.962963,0.999339,0.713303,0.814153,-0.933089,0.951720,-0.817906,0.006016,0.000040,0.005669
6972,6972,0,0,0,0,1,0,1,1,0,...,0.946429,1.000000,0.583019,0.809524,-1.051731,0.953042,-0.887412,0.001414,0.000039,0.001067
6973,6973,0,0,0,0,1,0,0,0,1,...,0.960979,0.999339,1.054972,0.567460,-1.138938,0.951058,-0.930502,-0.007182,0.000039,-0.007529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,8985,0,0,0,0,0,0,0,0,0,...,0.469577,0.837963,1.226772,0.822751,-0.707361,0.142857,-0.649616,0.002457,0.000155,0.001990
8986,8986,0,0,0,0,0,0,0,0,0,...,0.671958,0.837963,0.785877,0.805556,-0.715692,0.196098,-0.668289,0.002312,0.000156,0.001845
8987,8987,0,0,1,0,0,0,0,0,0,...,0.481481,0.787698,0.834898,0.823413,-0.723949,0.133929,-0.670946,0.002891,0.000156,0.002424
8988,8988,0,0,0,0,0,0,0,0,0,...,0.655423,0.783730,0.994026,0.851852,-0.684937,0.101852,-0.646265,0.008310,0.000156,0.007843


In [6]:
feature_groups = {
        'market_dynamics': ['M' + str(i) for i in range(1, 19)],
        'economic': ['E' + str(i) for i in range(1, 21)],
        'interest_rate': ['I' + str(i) for i in range(1, 10)],
        'price_valuation': ['P8', 'P9', 'P10', 'P12', 'P13'],
        'volatility': ['V' + str(i) for i in range(1, 14)],
        'sentiment': ['S' + str(i) for i in range(1, 13)],
        'dummy': ['D' + str(i) for i in range(1, 10)]
    }

In [7]:
def preprocessing(data, typ):
    main_featurs = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1',
                    'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19',
                    'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',
                    
                    'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9',
                    'M1', 'M10', 'M11', 'M12', 'M13','M14', 'M15', 'M16', 'M17', 'M18', 'M2', 
                    'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
                    
                    'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4',
                    'S5', 'S6', 'S7', 'S8', 'S9', 
                    
                    'V1', 'V10', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9',
                    "P9", "P8", "P10", "P12", "P13"]
    
    if typ == "train":
        data = data[main_featurs + ["forward_returns"]]
    else:
        data = data[main_featurs]
    
    for i in zip(data.columns, data.dtypes):
        data[i[0]].fillna(0, inplace=True)

    return data
    
train = h2o.H2OFrame(preprocessing(train, "train"))
test = h2o.H2OFrame(preprocessing(test, "test"))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [8]:
main_col = train.columns
main_col.remove('forward_returns')

In [9]:
aml = H2OAutoML(max_models = 30, seed = 42, max_runtime_secs = 1000)
aml.train(x=main_col, y='forward_returns', training_frame=train)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_1_AutoML_1_20250923_92956


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    20                 20                          4244                   5            10           7.45          11            14            12.3

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.00010263170208925117
RMSE: 0.010130730580232167
MAE: 0.007168031588611113
RMSLE: 0.010151568789406536
Mean Residual Deviance: 0.00010263170208925117

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 0.00011986771680825384
RMSE: 0.010948411611199764
MAE: 0.007715293898469947
RMSLE: 0.010967869251385945
Mean Residual Deviance: 0.00011986771680825384

Cross-Validation Metrics Summary: 
                        mean         sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  -----------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     nan          0            nan           nan           nan           nan           nan
loglikelihood           nan          0            nan           nan           nan           nan           nan
mae                     0.00777245   0.000181137  0.00797006    0.00757926    0.00759973    0.00793035    0.00778285
mean_residual_deviance  0.000121188  6.7507e-06   0.000127256   0.000111913   0.000116908   0.000122377   0.000127488
mse                     0.000121188  6.7507e-06   0.000127256   0.000111913   0.000116908   0.000122377   0.000127488
r2                      -0.0435552   0.0262705    -0.0260283    -0.0248988    -0.088894     -0.040952     -0.037003
residual_deviance       0.000121188  6.7507e-06   0.000127256   0.000111913   0.000116908   0.000122377   0.000127488
rmse                    0.0110051    0.000308173  0.0112808     0.0105789     0.0108124     0.0110624     0.0112911
rmsle                   0.0110249    0.000300106  0.0112746     0.0106101     0.0108502     0.0110576     0.011332

Scoring History: 
    timestamp            duration    number_of_trees    training_rmse    training_mae    training_deviance
--  -------------------  ----------  -----------------  ---------------  --------------  -------------------
    2025-09-23 09:30:12  3.462 sec   0                  0.010798         0.00761473      0.000116598
    2025-09-23 09:30:12  3.636 sec   5                  0.0105997        0.00747681      0.000112353
    2025-09-23 09:30:12  3.741 sec   10                 0.0104354        0.00737021      0.000108898
    2025-09-23 09:30:12  3.837 sec   15                 0.0102742        0.00726035      0.000105559
    2025-09-23 09:30:13  3.941 sec   20                 0.0101307        0.00716803      0.000102632

Variable Importances: 
variable    relative_importance    scaled_importance    percentage
----------  ---------------------  -------------------  --------------------
E19         0.008478515781462193   1.0                  0.0570459699393823
P10         0.008059006184339523   0.9505208685180602   0.054223384892236826
M11         0.00653789471834898    0.7711131154162296   0.04398889560189767
S11         0.005244705826044083   0.6185877294126577   0.03528793701694523
V5          0.005161426495760679   0.6087653344994479   0.034727608971993514
V6          0.004852023907005787   0.5722727930299393   0.032645856548312265
V9          0.0046279216185212135  0.5458410101258417   0.03113802985532084
P13         0.004467847757041454   0.5269610710414855   0.030061005417857285
M2          0.004334205761551857   0.5111986429309194   0.02916182241769026
M12         0.004306809976696968   0.50796744238107

In [10]:
def predict(test: pl.DataFrame) -> float:
    test = test.to_pandas()
    test = h2o.H2OFrame(preprocessing(test, "test"))
    raw_pred = aml.predict(test).as_data_frame()['predict'].values[0]
    return raw_pred

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
Parse progress: |

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


████████████████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
Parse progress: |

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


████████████████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
Parse progress: |

...

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


████████████████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
Parse progress: |

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


████████████████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
